# Kaggriculture, 800 paired games and the map wins

Eight hundred games, played locally on the competition engine, that put one question to the strongest
public agent on this tab: can a model of the market choose its tape better than the hand map its author
fitted on real matches? The agent is yhay81's two-shop tape router. It replays one of ten recorded
719-action histories and picks which one at step 144, by the first two shops the town has revealed. We
replaced only that choice, three different ways, and played every variant against the original on
identical seeds, seats and opponents.

The answer is no, and the reasons are worth more than a yes would have been. A market model that
reproduces season revenue to 76 coins when it knows the opponent's schedule, and to 4058 when it uses
the field average, still picks tapes that bank 3662 to 4832 coins less per game, and the shared-market
choice loses 11.0 points of win rate. Across 31 shop pairs, the gain the model expected and the bank
difference the games produced are unrelated: a Spearman correlation printed as -0.000.

What a fork gets: the exact market model, the shop-draw posterior, three runnable variants of the
router, and a paired evaluation harness that turns any change to an agent into a difference with a
confidence interval. All of it credits yhay81, whose router is used under Apache 2.0 with one table
replaced.

In [ ]:
import os, glob, json, math, statistics, collections, importlib.util
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
INK, ACC, MUT, GD = "#1b2a4a", "#c1440e", "#8a94a6", "#c29a22"

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, (name, os.listdir("/kaggle/input"))
    return hits[0]

results = pd.read_csv(find("results.csv"))
table1 = json.load(open(find("table.json")))["pairs"]
table2 = json.load(open(find("table2.json")))["pairs"]
valid = json.load(open(find("validate2.json")))
diverge = pd.read_csv(find("tape_divergence.csv"))
profiles = pd.read_csv(find("tape_profiles.csv"))
gaps = pd.read_csv(find("gap_breakdown.csv"))
spec = importlib.util.spec_from_file_location("yh", find("yhay81_main.py"))
yh = importlib.util.module_from_spec(spec); spec.loader.exec_module(yh)
print(f"games in results.csv: {len(results)}, agents: {dict(results['agent'].value_counts().sort_index())}")
print(f"opponents: {sorted(results['opponent'].unique())}, seeds {results['seed'].min()}..{results['seed'].max()}")
print(f"tapes in the router: {len(yh._ROUTES)}, branch steps {yh.FIRST_BRANCH_STEP} and {yh.SECOND_BRANCH_STEP}, "
      f"second-branch keys: {len(yh.SECOND_ROUTES)}")

## 1. One router, one decision

The router carries ten tapes and two branch points, at steps 72 and 144. The first branch sends seven
of the eight possible first shops to tape 0 and only a first YARN_STORE to tape 1. The second branch is
a 64-entry map from the ordered pair of the first two shops to a tape, and it sends 33 of the 64 pairs
to tape 0. The matrix below says why nothing later than step 144 can be a decision: tape 1 parts from
every other tape at step 73, and every other pair of tapes parts by step 150 at the latest. No two
tapes are still identical at step 216, when the third shop appears. A tape switched after 144 would be
replaying a history whose land, animals and plantings the farm does not have. So every variant in this
notebook changes the 64-entry map and nothing else.

The profile table is what the map chooses between. Tape 8 sells 296 units of wool and no carrots; tape
7 sells 79 units of wool and 423 of wheat; tape 5 sells the most milk and strawberry, 289 and 326 units.
Those differences are the whole lever.

In [ ]:
n = len(yh._ROUTES)
M = np.full((n, n), 0, dtype=int)
for _, r in diverge.iterrows():
    M[int(r.tape_a), int(r.tape_b)] = M[int(r.tape_b), int(r.tape_a)] = int(r.first_divergence_step)
print("step at which each pair of tapes first differs")
print("     " + "".join(f"{j:>5}" for j in range(n)))
for i in range(n):
    print(f"{i:>4} " + "".join(f"{M[i, j]:>5}" if i != j else "    ." for j in range(n)))
late = int((diverge["first_divergence_step"] >= 216).sum())
print(f"\npairs still identical at step 216: {late} of {len(diverge)}; latest divergence: step {diverge['first_divergence_step'].max()}")
print(f"first shop routes: {collections.Counter(yh.FIRST_ROUTES.values())}")
print(f"second shop routes: {collections.Counter(yh.SECOND_ROUTES.values())}")
print("\nunits each tape sells over the season, and the animals it buys")
print(profiles.to_string(index=False))

## 2. A market model exact to the coin, alone

The market model reproduces the engine's rules: the price response to inventory, the depth each shop
adds, the sell ticks, the shop unlocks, and the uniform draw of the shops not yet revealed, which enters
through the posterior our shop-draw notebook published. The table scores it on 120 games against real
opponents, one forced tape per game, revenue actually earned against revenue expected. The solo version,
which assumes the farm sells alone, is off by 52026 coins on average and always upward. Give the same
model the opponent's actual executed schedule and the error falls to 76 coins. Replace that schedule
with the pool's average and it is 4058. Integrate over the shop posterior, as the router must at step
144, and it is 14140. The market is shared; that is the whole difference between the two panels.

In [ ]:
V = pd.DataFrame(valid["rows"])
print(f"validation set: {valid.get('label')}; field schedule built from {valid.get('n_field_games')} games")
print("columns:", list(V.columns))
cols = [c for c in ("solo", "shared_exact", "shared_field", "shared_field_posterior") if c in V.columns]
print(f"\nvalidation games: {len(V)} (forced tape as farm 0 against a real opponent, revenue actually earned versus each model)")
print(f"{'model':24}{'MAE, coins':>12}{'mean error':>12}")
for c in cols:
    e = V[c] - V["actual"]
    print(f"{c:24}{e.abs().mean():>12.0f}{e.mean():>+12.0f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
for a, c, col, lab in ((ax[0], "solo", MUT, "solo seller model"), (ax[1], "shared_exact", INK, "shared market, opponent's real schedule")):
    a.scatter(V["actual"] / 1000, V[c] / 1000, s=18, color=col)
    lo, hi = min(V["actual"].min(), V[c].min()) / 1000, max(V["actual"].max(), V[c].max()) / 1000
    a.plot([lo, hi], [lo, hi], color=ACC, lw=1, ls="--")
    a.set_title(lab); a.set_xlabel("revenue actually earned, k coins")
ax[0].set_ylabel("revenue the model expected, k coins")
plt.tight_layout(); plt.savefig("fig1_model_validation.png", metadata={"Software": None}); plt.show()

## 3. What the model recommends

Two tables, one per model, each naming a tape for every one of the 64 ordered shop pairs. The table
built from executed sales disagrees with yhay81 on 48 pairs, 75.0 per cent of the draw mass, and sends
41 of them to tape 9, the wool tape. The table built from the shared-market model disagrees on 45
pairs, 70.3 per cent, and splits between tape 4 in 26 pairs and tape 5 in 21. yhay81's own map sends 33
pairs to tape 0. The product breakdown of the ten largest expected gaps is the tell. Where the model
prefers tape 6 its edge over tape 0 is 12016 coins of milk; where it prefers tape 9 the edge is 6971
coins of wool plus 3701 of fertilizer. Wool is the product our shop-draw notebook found without a buyer
in a third of towns, and that is the product the executed-sales table bets on.

In [ ]:
YH = yh.SECOND_ROUTES
def summary(tab, name):
    dis = [k for k in tab if tab[k]["posterior"] != YH.get(k, 0)]
    print(f"{name}: disagrees with yhay81 on {len(dis)} of {len(tab)} ordered shop pairs "
          f"({100 * len(dis) / len(tab):.1f} per cent of the draw mass)")
    print(f"   tapes the model picks: {dict(collections.Counter(v['posterior'] for v in tab.values()))}")
    return dis
print(f"tapes yhay81 picks: {dict(collections.Counter(YH.values()))}")
dis1 = summary(table1, "table from executed sales (variant B)")
dis2 = summary(table2, "table from the shared-market model (variant C)")
print("\nwhere the expected gain over yhay81's tape comes from, by product (ten largest gaps, executed-sales table)")
print(gaps.to_string(index=False))

## 4. Eight hundred paired games

Four opponents from the public tab, kaito48, v16rc5, k320 and closer_cleo; 25 seeds each; both seats;
the same seed, seat and opponent for the original and for every variant, so each difference below is a
paired difference. Two hundred games per agent, 800 in all.

Variant B, the executed-sales table, wins 2.5 points more often, an interval from -1.5 to +6.5 that
covers zero, and banks 4066 coins less, an interval from -7420 to -712 that does not. Variant C, the
shared-market table, loses 11.0 points, from -18.3 to -3.7, and 4832 coins; against kaito48 alone it
loses 28.0 points. Variant D, the wool tape in every town, is 1.0 point up and 3662 coins down.
Restricted to the pairs where a variant actually differs from yhay81, every interval widens and every
sign holds. The original wins 94.0 per cent of these games; the pool is not strong enough to separate
B and D from it on wins, and it is strong enough to show that all three earn less.

In [ ]:
key = lambda r: (int(r["seed"]), int(r["seat"]), r["opponent"])
A = {key(r): r for _, r in results[results["agent"] == "A"].iterrows()}
def chosen(v, pair):
    return {"B": table1[pair]["posterior"], "C": table2[pair]["posterior"], "D": 1}[v]
def ci(xs):
    mu = sum(xs) / len(xs); h = 1.96 * (statistics.stdev(xs) if len(xs) > 1 else 0) / math.sqrt(len(xs))
    return mu, mu - h, mu + h
LABEL = {"B": "B, table from executed sales", "C": "C, shared-market table", "D": "D, wool tape always"}
paired = {}
for v in ("B", "C", "D"):
    Vr = {key(r): r for _, r in results[results["agent"] == v].iterrows()}
    common = [k for k in Vr if k in A]
    U = collections.defaultdict(lambda: {"dw": [], "db": []}); UD = collections.defaultdict(lambda: {"dw": [], "db": []})
    n_dis = 0
    for k in common:
        a, b = A[k], Vr[k]; pair = f"{a['shop1']}__{a['shop2']}"
        dw, db = int(b["win"]) - int(a["win"]), float(b["bank"]) - float(a["bank"])
        U[(k[0], k[2])]["dw"].append(dw); U[(k[0], k[2])]["db"].append(db)
        if chosen(v, pair) != YH.get(pair, 0):
            n_dis += 1; UD[(k[0], k[2])]["dw"].append(dw); UD[(k[0], k[2])]["db"].append(db)
    wa = np.mean([int(A[k]["win"]) for k in common]); wv = np.mean([int(Vr[k]["win"]) for k in common])
    out = {}
    for lab, S in (("all", U), ("disagreeing", UD)):
        dw = [np.mean(x["dw"]) for x in S.values()]; db = [np.mean(x["db"]) for x in S.values()]
        out[lab] = {"units": len(dw), "win": ci(dw), "bank": ci(db)}
    paired[v] = {"games": len(common), "n_dis": n_dis, "winA": wa, "winV": wv, **out}
    print(f"{LABEL[v]}: {len(common)} games, {n_dis} on pairs where it differs from yhay81; win rate A {wa:.3f}, {v} {wv:.3f}")
    for lab in ("all", "disagreeing"):
        w, b = out[lab]["win"], out[lab]["bank"]
        print(f"   {lab:12} units {out[lab]['units']:>3}  win diff {100*w[0]:+5.1f} pp [{100*w[1]:+5.1f}, {100*w[2]:+5.1f}]"
              f"   bank diff {b[0]:+6.0f} [{b[1]:+6.0f}, {b[2]:+6.0f}]")
    byo = collections.defaultdict(lambda: {"dw": [], "db": []})
    for k in common:
        byo[k[2]]["dw"].append(int(Vr[k]["win"]) - int(A[k]["win"])); byo[k[2]]["db"].append(float(Vr[k]["bank"]) - float(A[k]["bank"]))
    for o, x in sorted(byo.items()):
        print(f"      {o:12} n {len(x['dw']):>3}  win diff {100*np.mean(x['dw']):+5.1f} pp  bank diff {np.mean(x['db']):+7.0f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
xs = np.arange(3); labs = ["B executed", "C shared", "D wool"]
for a, what, scale, yl in ((ax[0], "win", 100, "win rate, variant minus yhay81, points"), (ax[1], "bank", 1, "bank, variant minus yhay81, coins")):
    for i, v in enumerate(("B", "C", "D")):
        for j, (lab, col) in enumerate((("all", MUT), ("disagreeing", INK))):
            mu, lo, hi = paired[v][lab][what]
            a.errorbar(i + (j - 0.5) * 0.18, scale * mu, yerr=[[scale * (mu - lo)], [scale * (hi - mu)]], fmt="o", color=col, capsize=4,
                       label=(lab + " games") if i == 0 else None)
    a.axhline(0, color=ACC, lw=1, ls="--"); a.set_xticks(xs); a.set_xticklabels(labs); a.set_ylabel(yl)
ax[0].legend(loc="lower left", frameon=False)
plt.tight_layout(); plt.savefig("fig2_paired_differences.png", metadata={"Software": None}); plt.show()

## 5. Why an exact model still loses

Take variant C, the one built on the model that is exact when it knows the opponent. For each of the
31 shop pairs where C's tape differs from yhay81's and games were played, the table lists the gain the
model expected and the bank difference the games produced. Twenty-three of the 31 pairs went the wrong
way. The Spearman correlation between expected and realised prints as -0.000 with p = 0.998; Pearson
is +0.046 with p = 0.808. The model is not noisy. It is aimed at the wrong quantity.

A game is won on the bank difference against one particular opponent, and that opponent's schedule is
not the field average. It sells into the same shops at the same steps, and which tape survives that
contact depends on the opponent's tape, not on the town alone. An expectation over the field is the
right number for a lone seller and the wrong number for a duel. yhay81's map was fitted on the outcomes
of recorded matches, so it already contains the duel. A revenue model, however exact, does not.

In [ ]:
Cr = {key(r): r for _, r in results[results["agent"] == "C"].iterrows()}
per = collections.defaultdict(list)
for k, b in Cr.items():
    a = A.get(k)
    if a is None: continue
    pair = f"{a['shop1']}__{a['shop2']}"
    if table2[pair]["posterior"] != YH.get(pair, 0):
        per[pair].append(float(b["bank"]) - float(a["bank"]))
P = pd.DataFrame([{"pair": p, "n": len(v), "realised": np.mean(v),
                   "expected": table2[p]["revenue"][table2[p]["posterior"]] - table2[p]["revenue"][YH.get(p, 0)]} for p, v in per.items()])
P = P.sort_values("expected", ascending=False)
print(f"disagreeing pairs with games: {len(P)}; games behind them: {int(P['n'].sum())}")
print(f"{'pair':32}{'n':>4}{'model expected gain':>21}{'realised bank diff':>20}")
for _, r in P.iterrows():
    print(f"{r['pair']:32}{int(r['n']):>4}{r['expected']:>+21.0f}{r['realised']:>+20.0f}")
from scipy.stats import spearmanr, pearsonr
rs, ps = spearmanr(P["expected"], P["realised"]); rp, pp = pearsonr(P["expected"], P["realised"])
print(f"\nSpearman between expected gain and realised bank difference: {rs:+.3f} (p = {ps:.3f}); Pearson {rp:+.3f} (p = {pp:.3f})")
print(f"pairs where the model expected a gain and the bank fell: {int((P['realised'] < 0).sum())} of {len(P)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.scatter(P["expected"] / 1000, P["realised"] / 1000, s=14 + 4 * P["n"], color=INK)
ax.axhline(0, color=ACC, lw=1, ls="--")
ax.set_xlabel("gain the shared-market model expected from its tape, k coins")
ax.set_ylabel("bank difference actually realised, k coins")
ax.set_title("one point per disagreeing shop pair, size by games played")
plt.tight_layout(); plt.savefig("fig3_expected_vs_realised.png", metadata={"Software": None}); plt.show()

## 6. What this shows and what it does not

It shows, with paired intervals, that three model-driven replacements of the second-branch map earn
less than the map, and that one of them loses games. It shows the loss is not a modelling error, since
the same market model is exact to 76 coins with the opponent's schedule in hand. And it shows that the
model's expected gain per shop pair has no relation to the realised bank difference.

It does not show that the map is optimal. The pool is four public agents, not the live ladder; 25 seeds
per opponent give intervals of about 4 points either way on wins; and the original already wins 94.0
per cent of these games, so a gain on wins would be hard to see here even if it existed. A stronger
pool, and a variant that changes the tapes rather than the routing, are the two experiments this
harness makes cheap. What would overturn the conclusion is a variant that beats the original on wins
with an interval clear of zero, against a pool that includes the current top of the ladder.

## Fork notes

Everything is in the attached dataset. results.csv has one row per game: seed, seat, opponent, agent,
both banks, the win flag and the town's shops. table.json and table2.json hold the expected revenue of
all ten tapes for all 64 pairs under each model, with the tape each model picks and the tape yhay81
picks. validate2.json is the 120-game validation. revenue.py and revenue2.py are the market models,
shop_draw_model.py the posterior, eval_paired2.py the paired runner; the last carries local paths, so
point them at your own copies. main_posterior.py, main_shared.py and main_wool.py are the three
variants, each yhay81's file with one table replaced and a header that says so.

Credits. The router is
[yhay81's Public Match History Router](https://www.kaggle.com/code/yhay81/public-match-history-router-rating-2929-aug-30),
Apache 2.0. The opponents are public agents by kaitofukami, boatlee and raykkretzschmar, whose
reference agents also supplied closer_cleo. The posterior and the market-depth numbers come from two
earlier notebooks of mine on this competition,
[one town in three has no yarn store at all](https://www.kaggle.com/code/busyaprime/one-town-in-three-has-no-yarn-store-at-all)
and
[the best thing to farm has a market of 59 units](https://www.kaggle.com/code/busyaprime/the-best-thing-to-farm-has-a-market-of-59-units).